# Dos líneas base y cinco familias
Se entrenan primero DummyClassifier y la regla desgaste >= 200 o torque >= 60; luego logística, Random Forest, HistGradientBoosting, MLP y GaussianNB con valores por defecto salvo semilla. Misma preparación y tres folds para todos.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
import pandas as pd
from src.datos import DOCS, split
from src.pipeline import make_pipeline, PhysicalFeatures


In [2]:
from src.experimentos import baseline_candidates
rows = baseline_candidates()
display(pd.DataFrame(rows).drop(columns=["fold_ap","params"]))

fatal: not a git repository (or any parent up to mount point /)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


baseline: dummy: AP=0.0339 +/- 0.0002


🏃 View run dummy at: http://mlflow:5000/#/experiments/1/runs/ce60cc0838084d2e9b1fbb20cdf2d504
🧪 View experiment at: http://mlflow:5000/#/experiments/1


fatal: not a git repository (or any parent up to mount point /)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


baseline: business_rule: AP=0.1350 +/- 0.0289


🏃 View run business_rule at: http://mlflow:5000/#/experiments/1/runs/c1bd50d09be44c93bbd4f41b5f5e2173
🧪 View experiment at: http://mlflow:5000/#/experiments/1


... (15 outputs left out)

fatal: not a git repository (or any parent up to mount point /)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


ablation: hist_boosting_without_derived: AP=0.8021 +/- 0.0474


🏃 View run hist_boosting_without_derived at: http://mlflow:5000/#/experiments/1/runs/5d2ef8332c4c450bb5bfbd54457e953a
🧪 View experiment at: http://mlflow:5000/#/experiments/1


,name,phase,run_id,cv_ap_mean,cv_ap_std,train_ap,fold_train_ap_mean,cv_cost_per_1000,fit_seconds,cv_fit_seconds,gap_ap,warnings,learning_rate
0,dummy,baseline,ce60cc0838084d2e9b1fbb20cdf2d504,0.033875,0.000213,0.033875,0.033875,270999.883454,0.009024,0.031591,1.456823e-08,0,None
1,business_rule,baseline,c1bd50d09be44c93bbd4f41b5f5e2173,0.135011,0.028939,0.134074,0.134297,189627.482799,0.001063,0.004519,-9.368807e-04,0,None
2,logistic,candidato,7a4ebb4ac06c4414ba85910448265712,0.498286,0.069881,0.505527,0.506151,221000.975778,0.013154,0.035740,7.240666e-03,0,None
3,random_forest,candidato,9b518295ce8a40e98bcb622f9c48aac7,0.868115,0.041762,1.000000,1.000000,129812.425922,0.600027,1.313889,1.318851e-01,0,None
4,hist_boosting,candidato,5dac6d2cbb32455aad5a72c456841c89,0.878331,0.035684,1.000000,1.000000,121312.691315,0.128927,0.376170,1.216691e-01,0,0.1
5,mlp,candidato,dd89b550da4f4111b5af7f67f5edbdb0,0.753560,0.052316,0.868367,0.858493,164127.482002,1.725824,3.434191,1.148073e-01,4,constant
6,gaussian_nb,candidato,816e02646d994180847176189245deca,0.379369,0.019116,0.376087,0.380010,193688.865598,0.008974,0.026494,-3.282343e-03,0,None


In [3]:
ablation=json.loads((DOCS/'ablacion.json').read_text(encoding='utf-8'))
default=next(r for r in rows if r['name']=='hist_boosting')
display({'AP_con_features':default['cv_ap_mean'],'AP_sin_features':ablation['cv_ap_mean'], 'delta':default['cv_ap_mean']-ablation['cv_ap_mean'],'std_con_features':default['cv_ap_std']})

{'AP_con_features': 0.8783308704569471,
 'AP_sin_features': 0.8021225832900815,
 'delta': 0.07620828716686567,
 'std_con_features': 0.03568409005261371}

La brecha train AP menos CV AP ayuda a diagnosticar sobreajuste. La regla de negocio es una hipótesis del PSet anterior, pendiente de validación de un técnico. Las corridas se recuperan del checkpoint al reejecutar el notebook, sin duplicar búsqueda ni consumir el holdout.

In [4]:
from src.llm import run as prompting
display(prompting())

{'status': 'not_executed',
 'reason': 'Se requiere API key privada y LLM_MODEL explícito; no hay respuestas ni métricas inventadas.',
 'provider': 'openai',
 'protocol': '3 folds de entrenamiento; 20 filas por fold; ejemplos solo de su train; holdout nunca enviado',
 'prompt_template': 'Clasifica falla actual en una lectura sintética AI4I. Usa solo las mediciones disponibles. No infieras RUL ni eventos futuros. Devuelve únicamente JSON con probability entre 0 y 1. Las etiquetas de los ejemplos son 0 sin falla y 1 con falla. Ejemplos: {examples}. Lectura: {row}',
 'scope': 'La clasificación numérica sí admite prompting; esta comparación sigue pendiente.'}

El problema tabular puede formularse mediante prompting; se entrega un adaptador real para ChatGPT o Claude. Si no hay API key y modelo configurados, la salida indica not_executed y no afirma una evaluación inexistente. La prueba usa solo submuestras de folds de entrenamiento y nunca el holdout. Su presupuesto distinto no altera el ranking principal.